In [ ]:
#import libraries
import pandas as pd
import numpy as np

from pathlib import Path

In [ ]:
#finding csv files
csv_folder = Path("CSV")
csv_files = sorted(csv_folder.glob("*.csv"))

print(f"Number of CSV files: {len(csv_files)}")

for file in csv_files:
    print(file.name)

In [ ]:
#load datasets
customers = pd.read_csv(csv_folder / "olist_customers_dataset.csv")
geolocation = pd.read_csv(csv_folder / "olist_geolocation_dataset.csv")
order_items = pd.read_csv(csv_folder / "olist_order_items_dataset.csv")
payments = pd.read_csv(csv_folder / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(csv_folder / "olist_order_reviews_dataset.csv")
orders = pd.read_csv(csv_folder / "olist_orders_dataset.csv")
products = pd.read_csv(csv_folder / "olist_products_dataset.csv")
sellers = pd.read_csv(csv_folder / "olist_sellers_dataset.csv")
category_translation = pd.read_csv(csv_folder / "product_category_name_translation.csv")

In [ ]:
#inspecting dataset dimension
datasets = {
    "customers": customers,
    "geolocation": geolocation,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "orders": orders,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation
}

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]} columns")

In [ ]:
#inspect columns
for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print(list(df.columns))

In [ ]:
#check missing values
for name, df in datasets.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]

    print(f"\n{name.upper()}")

    if len(missing) == 0:
        print("No missing values")
    else:
        print(missing)

In [ ]:
# check duplicate rows
for name, df in datasets.items():
    duplicates = df.duplicated().sum()
    print(f"{name}: {duplicates:,} duplicate rows")

In [ ]:
#inspect data types
for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print(df.dtypes)

In [ ]:
#convert data columns
order_items["shipping_limit_date"] = pd.to_datetime(order_items["shipping_limit_date"])

reviews["review_creation_date"] = pd.to_datetime(reviews["review_creation_date"])
reviews["review_answer_timestamp"] = pd.to_datetime(reviews["review_answer_timestamp"])

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in date_columns:
    orders[column] = pd.to_datetime(orders[column])

In [ ]:
#verify data converted columns
print(order_items["shipping_limit_date"].dtype)
print(reviews[["review_creation_date", "review_answer_timestamp"]].dtypes)
print(orders[date_columns].dtypes)

In [ ]:
#check unique keys
key_checks = {
    "customers.customer_id": customers["customer_id"],
    "orders.order_id": orders["order_id"],
    "products.product_id": products["product_id"],
    "sellers.seller_id": sellers["seller_id"],
    "reviews.review_id": reviews["review_id"],
    "category_translation.product_category_name": category_translation["product_category_name"]
}

for name, column in key_checks.items():
    print(
        f"{name}: "
        f"{column.nunique():,} unique / "
        f"{len(column):,} total"
    )

In [ ]:
#check foireign key relationships
relationship_checks = {
    "orders -> customers": (~orders["customer_id"].isin(customers["customer_id"])).sum(),
    "order_items -> orders": (~order_items["order_id"].isin(orders["order_id"])).sum(),
    "order_items -> products": (~order_items["product_id"].isin(products["product_id"])).sum(),
    "order_items -> sellers": (~order_items["seller_id"].isin(sellers["seller_id"])).sum(),
    "payments -> orders": (~payments["order_id"].isin(orders["order_id"])).sum(),
    "reviews -> orders": (~reviews["order_id"].isin(orders["order_id"])).sum(),
    "products -> category_translation": (
        ~products["product_category_name"].dropna().isin(
            category_translation["product_category_name"]
        )
    ).sum()
}

for relationship, missing in relationship_checks.items():
    print(f"{relationship}: {missing:,} unmatched records")

In [ ]:
#dataset time range
print("First order:", orders["order_purchase_timestamp"].min())
print("Last order:", orders["order_purchase_timestamp"].max())
print("Total orders:", f"{len(orders):,}")
print("Unique customers:", f"{customers['customer_unique_id'].nunique():,}")
print("Unique products:", f"{products['product_id'].nunique():,}")
print("Unique sellers:", f"{sellers['seller_id'].nunique():,}")

In [ ]:
#order status distribution 
order_status = orders["order_status"].value_counts()

print(order_status)

In [ ]:
#visualize order status distribution 
import matplotlib.pyplot as plt

order_status.plot(kind="bar", figsize=(10, 5))

plt.title("Order Status Distribution")
plt.xlabel("Order Status")
plt.ylabel("Number of Orders")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
#calculate revenue overview
sales = order_items.merge(
    orders[["order_id", "order_purchase_timestamp", "order_status"]],
    on="order_id",
    how="left"
)

sales["year"] = sales["order_purchase_timestamp"].dt.year
sales["month"] = sales["order_purchase_timestamp"].dt.to_period("M")

print("Total revenue:", f"{sales['price'].sum():,.2f}")
print("Average item price:", f"{sales['price'].mean():,.2f}")
print("Total items sold:", f"{len(sales):,}")

In [ ]:
#monthly revenue
monthly_revenue = (
    sales.groupby("month")["price"]
    .sum()
    .reset_index()
)

monthly_revenue["month"] = monthly_revenue["month"].astype(str)

monthly_revenue

In [ ]:
#monthly revenue chart
plt.figure(figsize=(14, 6))

plt.plot(
    monthly_revenue["month"],
    monthly_revenue["price"],
    marker="o"
)

plt.title("Monthly Revenue")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
#top categories by revenue
category_sales = (
    order_items
    .merge(
        products[["product_id", "product_category_name"]],
        on="product_id",
        how="left"
    )
    .merge(
        category_translation,
        on="product_category_name",
        how="left"
    )
)

category_revenue = (
    category_sales
    .groupby("product_category_name_english", dropna=False)["price"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print(category_revenue)

In [ ]:
#top categories chart 
category_revenue.sort_values().plot(
    kind="barh",
    figsize=(10, 6)
)

plt.title("Top 10 Product Categories by Revenue")
plt.xlabel("Revenue")
plt.ylabel("Product Category")
plt.tight_layout()
plt.show()

In [ ]:
#order per customer 
customer_orders = (
    orders
    .merge(
        customers[["customer_id", "customer_unique_id"]],
        on="customer_id",
        how="left"
    )
    .groupby("customer_unique_id")["order_id"]
    .nunique()
)

print(customer_orders.describe())

In [ ]:
#repeat customers 
one_time_customers = (customer_orders == 1).sum()
repeat_customers = (customer_orders > 1).sum()

print("One-time customers:", f"{one_time_customers:,}")
print("Repeat customers:", f"{repeat_customers:,}")
print("Repeat customer rate:", f"{repeat_customers / len(customer_orders) * 100:.2f}%")
print("Maximum orders by one customer:", customer_orders.max())

In [ ]:
#payment types 
payment_types = payments["payment_type"].value_counts()

print(payment_types)

In [ ]:
#payment value by types
payment_value_by_type = (
    payments
    .groupby("payment_type")["payment_value"]
    .agg(["count", "sum", "mean"])
    .sort_values("sum", ascending=False)
)

payment_value_by_type

In [ ]:
#payment chart 
payment_types.plot(
    kind="bar",
    figsize=(9, 5)
)

plt.title("Payment Type Distribution")
plt.xlabel("Payment Type")
plt.ylabel("Number of Payments")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
#review score distribution 
review_scores = reviews["review_score"].value_counts().sort_index()

print(review_scores)

In [ ]:
#review score chart 
review_scores.plot(
    kind="bar",
    figsize=(8, 5)
)

plt.title("Review Score Distribution")
plt.xlabel("Review Score")
plt.ylabel("Number of Reviews")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
#delivery delay
delivery_analysis["delivery_status"] = np.select(
    [
        delivery_analysis["delivery_delay_days"].isna(),
        delivery_analysis["delivery_delay_days"] > 0
    ],
    [
        "Not delivered",
        "Late"
    ],
    default="On time or early"
)

print(delivery_analysis["delivery_status"].value_counts())

In [ ]:

delivery_reviews = delivery_analysis.merge(
    reviews[["order_id", "review_score"]],
    on="order_id",
    how="inner"
)

review_by_delivery = (
    delivery_reviews[
        delivery_reviews["delivery_status"] != "Not delivered"
    ]
    .groupby("delivery_status")["review_score"]
    .agg(["count", "mean"])
)

review_by_delivery

In [ ]:
#delivery time
delivery_time = orders[
    ["order_purchase_timestamp", "order_delivered_customer_date"]
].dropna().copy()

delivery_time["delivery_days"] = (
    delivery_time["order_delivered_customer_date"]
    - delivery_time["order_purchase_timestamp"]
).dt.total_seconds() / 86400

print(delivery_time["delivery_days"].describe())

In [ ]:
#average delivery time
print("Average delivery time:", f"{delivery_time['delivery_days'].mean():.2f} days")
print("Median delivery time:", f"{delivery_time['delivery_days'].median():.2f} days")

In [ ]:
#customers by state 
customers_by_state = (
    customers["customer_state"]
    .value_counts()
    .head(10)
)

print(customers_by_state)

In [ ]:
#customers by state chart 
customers_by_state.sort_values().plot(
    kind="barh",
    figsize=(9, 6)
)

plt.title("Top 10 States by Number of Customers")
plt.xlabel("Number of Customers")
plt.ylabel("State")
plt.tight_layout()
plt.show()

In [ ]:
#revenue by customer state 
state_revenue = (
    order_items
    .merge(
        orders[["order_id", "customer_id"]],
        on="order_id",
        how="left"
    )
    .merge(
        customers[["customer_id", "customer_state"]],
        on="customer_id",
        how="left"
    )
    .groupby("customer_state")["price"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print(state_revenue)

In [ ]:
#revenue by state chart 
state_revenue.sort_values().plot(
    kind="barh",
    figsize=(9, 6)
)

plt.title("Top 10 States by Revenue")
plt.xlabel("Revenue")
plt.ylabel("State")
plt.tight_layout()
plt.show()

In [ ]:
# EDA summary
print("=== EDA SUMMARY ===")

print(f"\nDataset period: {orders['order_purchase_timestamp'].min()} to {orders['order_purchase_timestamp'].max()}")

print(f"\nTotal orders: {len(orders):,}")
print(f"Unique customers: {customers['customer_unique_id'].nunique():,}")
print(f"Unique products: {products['product_id'].nunique():,}")
print(f"Unique sellers: {sellers['seller_id'].nunique():,}")

print(f"\nTotal product revenue: {sales['price'].sum():,.2f}")
print(f"Average item price: {sales['price'].mean():,.2f}")
print(f"Total items sold: {len(sales):,}")

print(f"\nOne-time customers: {one_time_customers:,}")
print(f"Repeat customers: {repeat_customers:,}")
print(f"Repeat customer rate: {repeat_customers / len(customer_orders) * 100:.2f}%")

print(f"\nAverage delivery time: {delivery_time['delivery_days'].mean():.2f} days")
print(f"Median delivery time: {delivery_time['delivery_days'].median():.2f} days")

print(f"\nLate delivery average review: {review_by_delivery.loc['Late', 'mean']:.2f}")
print(f"On-time delivery average review: {review_by_delivery.loc['On time or early', 'mean']:.2f}")

print(f"\nTop customer state: {customers_by_state.index[0]}")
print(f"Customers in top state: {customers_by_state.iloc[0]:,}")
print(f"Top state by revenue: {state_revenue.index[0]}")
print(f"Revenue in top state: {state_revenue.iloc[0]:,.2f}")

In [ ]:
#duplicate review IDs
duplicate_review_ids = reviews[
    reviews["review_id"].duplicated(keep=False)
].sort_values("review_id")

print("Rows with duplicated review IDs:", len(duplicate_review_ids))
print("Duplicated review IDs:", duplicate_review_ids["review_id"].nunique())

duplicate_review_ids.head(20)

In [ ]:
#checking review composite key
duplicate_review_order = reviews.duplicated(
    subset=["review_id", "order_id"]
).sum()

print("Duplicate review_id + order_id pairs:", duplicate_review_order)